[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/ab%20db/antibody_db_guide/06_structure/06_structure_lab.ipynb)


# 06 — 구조예측 (IgFold 직접 실행)

> 본문: [`06_structure.md`](06_structure.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 16초** (실측 — 도구가 도는 시간만. clone·pip·apt 설치 시간은 빼고 잰 값이에요)
> 코랩 무료 런타임은 코어가 2개뿐이라 설치에 1~2분, 무거운 예측 단계는 몇 배까지 더 걸릴 수 있어요 — 정상입니다.

**이 노트북은 도구를 직접 돌립니다.** IgFold 를 **직접 돌려** Fv 구조를 예측하고(`my_run/`), 커밋된 예측 구조와 CA-RMSD 로 대조해요.
내 결과는 `my_run/` 에 쌓이고 커밋된 `data/` 는 대조군이에요 — 어느 단계가 실패해도 `resolve()` 가 `data/` 로 폴백해 뒤 절이 계속 돌아갑니다.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

Colab 은 이 셀 하나로 끝나고, 로컬은 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "06_structure"
PIP_PKGS = "pandas matplotlib biopython igfold anarci abnumber py3Dmol gemmi"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True    # ANARCI 계열은 hmmscan(HMMER) 실행파일이 필요해요 (pip 로는 안 깔려요)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후엔 cwd 아래만 깊이 3까지 — 위로 올라가 rglob 하면 Colab 에서 / 전체를 뒤집니다.
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False
_APT = []                                # 필요한 시스템 패키지를 모아 apt 를 한 번만 돌립니다

if shutil.which("hmmscan") is None:      # ANARCI 가 부르는 실행파일 — pip 로는 안 깔려요
    if IN_COLAB:
        _APT.append("hmmer")
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")           # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨집니다

if _APT:
    _run("apt-get -qq update", quiet=True)   # 인덱스가 낡으면 install 이 404 로 죽습니다
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), quiet=True)

# IgFold 체크포인트에는 옛 transformers 의 토크나이저가 pickle 돼 있어, 5.x 로는 unpickle 이 실패해요.
_ver = subprocess.run([sys.executable, "-c", "import transformers;print(transformers.__version__)"],
                      capture_output=True, text=True).stdout.strip()
if not _ver.startswith("4."):
    print(f"[transformers {_ver or 'none'} → 4.36.2] IgFold 호환 버전으로 맞춥니다")
    _run(f'"{sys.executable}" -m pip -q install "transformers==4.36.2"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

QUIET_WARNINGS = True   # 라이브러리 내부 경고 소음을 끕니다. 다 보고 싶으면 False 로 두세요

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요.

    igfold·biopython·transformers 는 **자기 패키지 소스 줄번호**가 찍힌
    DeprecationWarning/FutureWarning 을 실제 결과보다 길게 쏟아내요. 그게 성공 메시지를
    덮어버려 실패로 오해하게 만들어서, 기본으로 PYTHONWARNINGS=ignore 를 걸어 둡니다.
    도구가 직접 print 하는 안내·에러는 그대로 남아요(warnings 모듈만 막는 거예요)."""
    args = [str(a) for a in args]
    # 절대경로를 그대로 찍으면 한 줄이 화면을 넘겨 정작 중요한 옵션이 안 보여요.
    # /usr/bin/python3 → python, /…/scripts/x.py → scripts/x.py 로 줄여서 보여줍니다.
    def _short(s):
        if s == PY:
            return "python"
        return "scripts/" + s[len(str(SCRIPTS)) + 1:] if s.startswith(str(SCRIPTS) + os.sep) else s
    print("$", " ".join(_short(a) for a in args))
    env = {**os.environ, "PYTHONWARNINGS": "ignore"} if QUIET_WARNINGS else None
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout, env=env)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — IgFold 로 Fv 예측 (본문 6.1)

```bash
python scripts/run_igfold_demo.py --fasta data/demo_mab.fa --out my_run/demo_antibody_igfold.pdb
```
IgFold 는 항체 언어모델(AntiBERTy) + graph network 로 backbone 을 예측하고, **B-factor 컬럼에
잔기별 예측오차(Å)** 를 적어 줘요. 실행 중 나오는 함정 두 가지는 스크립트가 처리합니다.

- `torch ≥ 2.6` 의 `weights_only=True` → 체크포인트 로드 실패 → `weights_only=False` 로 감싸기
- 최신 `transformers`(5.x) → 체크포인트 unpickle 실패 → 부트스트랩이 **`transformers==4.36.2`** 로 맞춰 줌

> `RUN_IGFOLD = False` 로 두면 예측을 건너뛰고 커밋된 PDB(`data/`)로 나머지 절을 진행해요.

In [ ]:
RUN_IGFOLD = True     # False 면 예측을 건너뛰고 커밋본(data/)으로 진행

# 걸리는 시간은 코어 수에 정직하게 비례해요 — 멀티코어 CPU 9초, 코랩 무료(2 vCPU) 40~60초.
# AntiBERTy + IgFold 모델 4개를 차례로 돌리는 단계라, 1분 가까이 걸려도 실패가 아니에요.

if RUN_IGFOLD:
    run_tool([PY, SCRIPTS/"run_igfold_demo.py",
              "--fasta", "data/demo_mab.fa",
              "--out", "my_run/demo_antibody_igfold.pdb"])
else:
    print("RUN_IGFOLD=False → 예측을 건너뛰고 레퍼런스 PDB 로 진행합니다.")

## 2) 내 결과 확인 — 사슬별 신뢰도와 최대오차 잔기 (본문 6.2)

본문이 "봉우리는 CDR-H3" 라고 읽으라 하니, **최대오차가 몇 번 잔기인지**까지 뽑아서 확인해요.
기준선은 본문 그래프와 같은 **1 Å** 입니다.

In [ ]:
import pandas as pd
from IPython.display import display

CONF = 1.0      # 본문 6.2 그래프의 빨간 기준선 (< 1 Å 이면 신뢰)

# 번호는 IgFold 가 do_renum=True 로 붙인 Chothia 번호(insertion code 포함).
# CDR 구간은 Kabat 정의 범위로 판정해요.
CDR_RANGES = {"H": [("CDR-H1", 31, 35), ("CDR-H2", 50, 65), ("CDR-H3", 95, 102)],
              "L": [("CDR-L1", 24, 34), ("CDR-L2", 50, 56), ("CDR-L3", 89, 97)]}

def region_of(chain, num):
    """잔기번호 → CDR 이름 또는 FR."""
    for name, lo, hi in CDR_RANGES.get(chain, []):
        if lo <= num <= hi:
            return name
    return "FR"

pdb = resolve("demo_antibody_igfold.pdb")
res, atom_names, n_atoms = {}, set(), 0
for line in open(pdb):
    if not line.startswith("ATOM"):
        continue
    n_atoms += 1
    name = line[12:16].strip()
    atom_names.add(name)
    if name == "CA":
        # (번호, insertion code, 예측오차) — 파일에 적힌 순서가 곧 서열 순서예요
        res.setdefault(line[21], []).append((int(line[22:26]), line[26].strip(), float(line[60:66])))

peaks, summary = [], []
for chain, v in sorted(res.items()):
    err = [b for _, _, b in v]
    num, ins, top = max(v, key=lambda r: r[2])
    over = [(f"{chain}{n}{i}", b, region_of(chain, n)) for n, i, b in sorted(v, key=lambda r: -r[2])
            if b > CONF]
    peaks += over
    summary.append({"사슬": chain, "잔기 수": len(v),
                    "평균 오차 (Å)": round(sum(err)/len(err), 2),
                    "최대 오차 (Å)": round(top, 2),
                    "최대인 잔기": f"{chain}{num}{ins}",
                    "그 잔기의 구간": region_of(chain, num),
                    f"{CONF:g} Å 초과": f"{len(over)}개"})

print("사슬별 요약 — 평균이 낮고 최대만 튀면 '대부분 믿을 만한데 한 loop 만 불확실' 이라는 뜻이에요.")
display(pd.DataFrame(summary))

if peaks:
    print(f"\n{CONF:g} Å 을 넘는 잔기 {len(peaks)}개 (오차 큰 순)")
    display(pd.DataFrame([{"잔기": lbl, "예측오차 (Å)": round(b, 2), "구간": r,
                           "막대": "█" * max(1, round(b / CONF * 4))}
                          for lbl, b, r in sorted(peaks, key=lambda x: -x[1])]))

hot = sorted({r for _, _, r in peaks})
print(f"\n판정 — {CONF:g} Å 을 넘는 잔기 {len(peaks)}개, 소속 구간 " + (" · ".join(hot) or "없음"))
if peaks and all(r != "FR" for _, _, r in peaks):
    print("→ 불확실한 곳이 CDR loop 로만 몰렸어요. framework 좌표는 쓰고 그 loop 만 교차검증하세요(본문 6.4).")
elif peaks:
    print("→ CDR 밖에도 1 Å 을 넘는 곳이 있어요. 그 구간부터 확인하세요.")
else:
    print("→ 전 구간이 1 Å 아래예요.")

print(f"\n원자 {n_atoms}개 · 원자 종류 " + " · ".join(sorted(atom_names)))
print("스크립트가 do_refine=False 로 돌려서 side chain 이 없는 backbone(+CB) 모델이에요. "
      "side chain 원자 접촉이 필요한 분석(Ch.07 interface·Ch.08)에 이 PDB 를 그대로 넣으면 안 됩니다.")

## 3) 내 결과 확인 — 신뢰도 프로파일 그래프 (본문 6.2)

잔기 번호에 **insertion code** 가 붙어 있어요(H52A·H82A~C·H100A~C). 번호를 정수로만 쓰면
H100 / H100A / H100B / H100C 가 **한 x 좌표에 겹쳐** 봉우리 폭이 뭉개집니다 — 하필 본문이
"CDR-H3 구간" 이라고 읽으라는 바로 그 자리예요. 그래서 여기서는 **사슬 안 순차 인덱스**로 그리고,
1 Å 을 넘는 점에는 실제 잔기 라벨을 달아요.

In [ ]:
import antibody_viz                    # import 만으로 한글 폰트 등록 + 팔레트 재사용
import matplotlib.pyplot as plt
from IPython.display import Image, display

png = "my_run/06_structure_confidence.png"
palette = {"H": antibody_viz.C_PURPLE, "L": antibody_viz.C_ORANGE}

fig, ax = plt.subplots(figsize=(11, 5))
for chain, v in sorted(res.items()):
    ys = [b for _, _, b in v]
    ax.plot(range(len(ys)), ys, marker="o", ms=3, lw=1.4,
            color=palette.get(chain, antibody_viz.C_TEAL),
            label=f"chain {chain} (mean {sum(ys)/len(ys):.2f} Å)")
    for k, (num, ins, b) in enumerate(v):
        if b > CONF:
            ax.annotate(f"{chain}{num}{ins}", (k, b), fontsize=7, ha="center", va="bottom")
ax.axhline(CONF, ls="--", color=antibody_viz.C_THR, lw=1.5, label=f"confident (<{CONF:g} Å)")
ax.set_title("IgFold confidence — demo mAb (Fv, 내 실행 결과)", fontsize=14, fontweight="bold")
ax.set_xlabel("Residue index (사슬 내 순서 — insertion code 포함)", fontsize=11)
ax.set_ylabel("Predicted error / B-factor (Å)", fontsize=11)
ax.grid(alpha=0.25); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(png, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(png))

flat = {chain: max(b for _, _, b in v) for chain, v in res.items()}
print("사슬별 최대 예측오차 —", " · ".join(f"{k} 사슬 {v:.2f} Å" for k, v in sorted(flat.items())))
print(f"→ 라벨이 달린 봉우리가 {len(peaks)}개. 나머지 잔기는 전부 {CONF:g} Å 선 아래라 "
      f"backbone 골격은 그대로 써도 됩니다.")

## 4) 레퍼런스 대조 — 커밋된 예측 구조와 얼마나 같은가 (본문 6.2b)

같은 서열·같은 모델이라도 실행 환경(BLAS·스레드 수)에 따라 좌표가 소수점 단위로 흔들릴 수 있어요.
그래서 "완전 동일" 이 아니라 **CA 좌표 RMSD** 와 사슬별 예측오차 통계로 비교합니다.

In [ ]:
import pathlib

RMSD_OK = 0.01      # 이 아래면 사실상 같은 구조 (본문 6.2b 실측 0.002 Å)
mine_p = pathlib.Path("my_run/demo_antibody_igfold.pdb")

if mine_p.exists():
    import pandas as pd
    from Bio.PDB import PDBParser, Superimposer
    parser = PDBParser(QUIET=True)

    def ca(path):
        return [a for a in parser.get_structure("x", str(path)).get_atoms() if a.get_id() == "CA"]

    a, b = ca(mine_p), ca("data/demo_antibody_igfold.pdb")
    print(f"CA 원자 수 — 내 결과 {len(a)} / 레퍼런스 {len(b)}")

    # 내 예측 vs 커밋 예측을 사슬별로 나란히 — 눈으로 바로 대조되게 표로 냅니다.
    stat = {}
    for label, path in [("내 결과", mine_p), ("레퍼런스", "data/demo_antibody_igfold.pdb")]:
        d = {}
        for line in open(path):
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                d.setdefault(line[21], []).append(float(line[60:66]))
        for k, v in d.items():
            stat.setdefault(k, {})[label] = (sum(v) / len(v), max(v))
    display(pd.DataFrame([
        {"사슬": k,
         "내 결과 (평균)":   f"{s.get('내 결과',   (float('nan'),))[0]:.2f} Å",
         "레퍼런스 (평균)":  f"{s.get('레퍼런스',  (float('nan'),))[0]:.2f} Å",
         "내 결과 (최대)":   f"{s.get('내 결과',   (0, float('nan')))[1]:.2f} Å",
         "레퍼런스 (최대)":  f"{s.get('레퍼런스',  (0, float('nan')))[1]:.2f} Å"}
        for k, s in sorted(stat.items())]))

    if len(a) == len(b):
        sup = Superimposer(); sup.set_atoms(b, a)
        print(f"\nCA-RMSD (내 예측 vs 커밋 예측) = {sup.rms:.3f} Å")
        if sup.rms < RMSD_OK:
            print(f"→ {RMSD_OK:g} Å 아래 = 사실상 같은 구조. IgFold 예측은 결정론적으로 재현됩니다.")
        else:
            print(f"→ {RMSD_OK:g} Å 를 넘었어요. transformers 버전·CPU/GPU 여부부터 맞춰 보세요.")
    else:
        print("\nCA 개수가 달라 RMSD 를 건너뜁니다 — 입력 FASTA 가 같은지 확인하세요.")
else:
    print("my_run 예측이 없어 대조를 건너뜁니다 (RUN_IGFOLD=False 였거나 실행 실패).")

## 5) 3D 구조 렌더 — 예측오차로 색칠한 인라인 뷰어 (본문 6.2)

**내가 방금 예측한 구조를 노트북 안에서 바로 돌려 봅니다.** `py3Dmol`(3Dmol.js)은 pip 로 깔리고
Colab 에서도 그대로 떠요 — 드래그로 회전, 휠로 확대됩니다.

- **cartoon 색 = 3절 그래프와 똑같은 값**인 잔기별 예측오차(B-factor).
  `roygb` 그라디언트를 `min>max` 로 뒤집어 넣어 **빨강 = 오차 큼(2 Å) → 노랑 → 초록 → 파랑 = 오차 작음(0 Å)** 이에요.
- **빨간 스틱 = 3절에서 기준선 1 Å 을 넘긴 바로 그 잔기들.** 여기서 다시 세지 않고 2절이 만든 `res`·`CONF` 를 그대로 씁니다.
- 강조는 **별도 모델로 얹어요** — `H100A`·`H100B` 처럼 insertion code 가 붙은 잔기가 선택 문법에서 새는 걸 원천 차단합니다.

> IgFold 는 `do_refine=False` 로 돌아 **backbone + CB 만** 있어요. 스틱이 짧게 보이는 건 정상입니다.

In [ ]:
import importlib.util, sys, pathlib
for _pkg in ("py3Dmol", "gemmi"):        # 부트스트랩이 이미 깔지만, 이 셀만 따로 돌릴 때 대비
    if importlib.util.find_spec(_pkg) is None:
        _run(f'"{sys.executable}" -m pip -q install {_pkg}')
import py3Dmol, gemmi

view_src = pathlib.Path(pdb)               # 2절 resolve() 가 고른 바로 그 PDB
assert view_src.exists(), f"{view_src} 가 없어요 — 1~2절을 먼저 실행하세요."
WHICH = "내 결과 (my_run/)" if view_src.parts[0] == "my_run" else "레퍼런스 (data/ 커밋본)"
print(f"[3D 뷰어] 표시 대상 = {WHICH} — {view_src}")

pdb_text = gemmi.read_structure(str(view_src)).make_pdb_string()   # PDB/CIF 무엇이든 PDB 문자열로 통일
BMAX = 2.0                                 # 색 스케일 상한(Å)

# 3절에서 CONF(=1 Å)를 넘긴 그 집합 — 하드코딩이 아니라 계산된 값 그대로
hot_keys = {(ch, n, ins) for ch, v in res.items() for n, ins, b in v if b > CONF}

def sub_pdb(text, keys):
    """(chain, resSeq, iCode) 집합에 드는 원자 줄만 뽑아 작은 PDB 를 만든다 (insertion code 안전)."""
    return "".join(L for L in text.splitlines(keepends=True)
                   if L.startswith(("ATOM", "HETATM"))
                   and (L[21], int(L[22:26]), L[26].strip()) in keys)

view = py3Dmol.view(width=820, height=560)
view.addModel(pdb_text, "pdb")
view.setStyle({"model": -1},               # model:-1 = 방금 추가한 모델
              {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb",
                                           "min": BMAX, "max": 0.0}}})
hot_text = sub_pdb(pdb_text, hot_keys)
if hot_text:
    view.addModel(hot_text, "pdb")
    view.setStyle({"model": -1}, {"stick":  {"radius": 0.28, "color": "red"},
                                  "sphere": {"radius": 0.45, "color": "red"}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

def lab(keys):                             # H96, H97 ... 번호 순으로 (문자열 정렬이면 H100 이 H96 앞에 와요)
    return [f"{c}{n}{i}" for c, n, i in sorted(keys)]

print(f"빨갛게 강조한 잔기 {len(hot_keys)}개 — " + " · ".join(lab(hot_keys))
      + f" (원자 {len(hot_text.splitlines())}개 — 0 이면 강조가 안 그려진 것)")
missing = hot_keys - {(L[21], int(L[22:26]), L[26].strip()) for L in hot_text.splitlines()}
print("좌표에서 못 찾은 잔기 — " + (" · ".join(lab(missing)) or "없음"))
print("\n판정 — ① 빨간 스틱이 한 loop 에 몰려 있나 ② 그 자리 cartoon 도 빨강/노랑인가 "
      "③ 3절 그래프에서 라벨이 붙은 잔기와 같은 이름인가.")
print("셋이 맞으면 2D 그래프와 3D 구조가 같은 결론을 말하는 거예요. 돌려 보면 그 loop 가 "
      "구조 바깥으로 튀어나온 끝단이라는 것도 보입니다 — CDR-H3 가 길고 자유로워 예측이 어려운 이유예요.")

## 5b) (선택) PyMOL 고해상도 정지 이미지

**PyMOL 은 pip 로 설치되지 않아요**(Colab 미지원) — 위 5절 인라인 뷰어가 주 시각화이고, 이 절은 덤이에요.
로컬에 PyMOL 이 있으면 **내 예측**을 다시 렌더해 `my_run/06_structure_3d.png` 로 저장하고,
없으면 저장소에 커밋된 렌더 이미지를 **레퍼런스라고 밝히고** 보여줍니다.
커밋된 `scripts/render_06_structure.pml` 은 건드리지 않고 **경로만 바꾼 사본**을 `my_run/` 에 만들어 써요.

In [ ]:
import pathlib, shutil, subprocess
from IPython.display import Image, display

def repath_pml(src, load_path, png_path, selections=None):
    """커밋된 .pml 의 load/png 경로(필요하면 select 문)만 바꾼 사본 텍스트를 만든다."""
    out = []
    for line in pathlib.Path(src).read_text().splitlines():
        s = line.strip()
        rest = ("," + line.split(",", 1)[1]) if "," in line else ""
        if s.startswith("load "):
            line = f"load {load_path}{rest}"
        elif s.startswith("png "):
            line = f"png {png_path}{rest}"
        elif selections and s.startswith("select "):
            key = s.split(",", 1)[0].split()[1]
            if key in selections:
                line = f"select {key}, {selections[key]}"
        out.append(line)
    return "\n".join(out) + "\n"

committed = "06_structure_3d.png"                 # 커밋본 (덮어쓰지 않아요)
mine_png = (MYRUN/"06_structure_3d.png").resolve()
shown, origin = committed, "레퍼런스(커밋된 렌더) — 내 결과가 아닙니다"
note = "PyMOL 없음(예: Colab) → 커밋된 렌더를 표시합니다. 내 구조는 5절 뷰어에서 보세요."

if shutil.which("pymol"):
    tmp_pml = MYRUN/"render_06_structure.my_run.pml"
    tmp_pml.write_text(repath_pml(ADV_ROOT/"scripts"/"render_06_structure.pml",
                                  pathlib.Path(pdb).resolve(),      # 2절이 고른 그 PDB
                                  mine_png))
    try:
        subprocess.run(["pymol", "-cq", str(tmp_pml)], cwd=str(ADV_ROOT),
                       check=True, capture_output=True, text=True, timeout=180)
        shown, origin = str(mine_png), f"내 구조 재렌더 ({view_src})"
        note = f"PyMOL 재렌더 완료 → {mine_png}"
    except Exception as e:
        note = f"PyMOL 실행 실패({type(e).__name__}) → 커밋된 렌더를 표시합니다."

print(note)
display(Image(shown))
print(f"판정 — 표시한 이미지 = {shown}  |  출처 = {origin}")

## 6) 정리 — 예측 구조를 어디까지 믿나 (본문 6.3)

- 이건 **예측 구조지 실험 구조가 아니에요.** 보고서에서 결정구조처럼 단정하면 안 됩니다.
- 방금 실측한 대로 1 Å 을 넘는 불확실성은 **CDR loop 한 곳에 몰려 있어요** — 이 loop 는
  ImmuneBuilder 등으로 교차검증하고(본문 6.4), 갈리면 후보 순위를 내려요.
- 항체 단독 구조는 **항원 결합 pose 를 보장하지 않아요** → 결합은 복합체 구조로 봅니다(Ch.07).

> 다음 → 본문 [07. interface 분석](../07_interface/07_interface.md)